In [54]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
data=fetch_california_housing()

In [56]:
data

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]]),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': '.. _california_housing_dataset:\n

In [57]:
df=pd.DataFrame(data.data,columns=data.feature_names)

In [58]:
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32


In [59]:
df['Target']=data.target

In [60]:
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


In [61]:
x_train,x_test,y_train,y_test=train_test_split(df.drop('Target',axis=1),df['Target'],test_size=0.2,random_state=42)

In [62]:
std=StandardScaler()

In [63]:
x_train_s=std.fit_transform(x_train)
x_test_s=std.transform(x_test)

to ann

In [64]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping as eralyStopping
from tensorflow.keras.layers import Dropout

In [65]:
def build_model(hp):
    model=Sequential()
    inp=0

    layer=hp.Int('Layer',min_value=1,max_value=10)
    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])
    loss=hp.Choice('Loss',['mean_squared_error','mean_absolute_error'])

    for i in range(layer):

      neurons = hp.Int(f'neurons_{i}', min_value=8, max_value=100, step=8)
      activation = hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid'])

      if inp==0:
        model.add(Dense(neurons,activation=activation,input_dim=x_train_s.shape[1]))
        model.add(Dropout(0.2))
        inp=1
      else:
        model.add(Dense(neurons,activation=activation))
        model.add(Dropout(0.2))
    model.add(Dense(1,activation='linear'))
    model.compile(optimizer=optimizer,loss=loss)
    return model

In [66]:
!pip install keras-tuner


In [67]:
import keras_tuner as kt

In [68]:
tuner=kt.RandomSearch(build_model,objective='val_loss',max_trials=5,directory='california_housing',project_name='california_housing')

Reloading Tuner from california_housing/california_housing/tuner0.json


In [69]:
callback=eralyStopping(monitor='val_loss',patience=10)
tuner.search(x_train_s,y_train,epochs=50,validation_data=(x_test_s,y_test),callbacks=[callback])

In [70]:
tuner.get_best_hyperparameters()[0].values

{'Layer': 9,
 'optimizer': 'rmsprop',
 'Loss': 'mean_squared_error',
 'neurons_0': 40,
 'activation_0': 'tanh',
 'neurons_1': 32,
 'activation_1': 'relu',
 'neurons_2': 64,
 'activation_2': 'sigmoid',
 'neurons_3': 24,
 'activation_3': 'sigmoid',
 'neurons_4': 64,
 'activation_4': 'sigmoid',
 'neurons_5': 88,
 'activation_5': 'relu',
 'neurons_6': 80,
 'activation_6': 'tanh',
 'neurons_7': 64,
 'activation_7': 'tanh',
 'neurons_8': 40,
 'activation_8': 'relu'}

In [71]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [72]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 40)             │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 24)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 88)             │         5,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 88)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 80)             │         7,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         5,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 40)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            41 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,609 (107.85 KB)

 Trainable params: 27,609 (107.85 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
data=model.fit(x_train_s,y_train,epochs=50,validation_data=(x_test_s,y_test),callbacks=[callback],batch_size=32)

Epoch 1/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.3643 - val_loss: 0.3264
Epoch 2/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.3717 - val_loss: 0.3298
Epoch 3/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.3705 - val_loss: 0.3364
Epoch 4/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.3658 - val_loss: 0.3169
Epoch 5/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3742 - val_loss: 0.3153
Epoch 6/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.3596 - val_loss: 0.3210
Epoch 7/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3806 - val_loss: 0.3162
Epoch 8/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3844 - val_loss: 0.3126
Epoch 9/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3604 - val_loss: 0.3187
Epoch 10/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3568 - val_loss: 0.3408
Epoch 11/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3560 - val_loss: 0.3159
Epoch 12/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/ste

In [74]:
y_pred=model.predict(x_test_s)

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [75]:
from sklearn.metrics import r2_score

In [76]:
r2_score(y_test,y_pred)

0.7611896738655838